# Stock data for our ML project

We all use the same **nine features**, **next-day return target**, and **date splits**. Each model predicts tomorrow's return. We then rank those predictions for each day.

Run the cells from top to bottom in a clone of this repo. In Colab, first run `!git clone https://github.com/Simon666Z/financial-engineering-club-fall26.git`, then `%cd financial-engineering-club-fall26`.

The stock list comes from when we downloaded it, so it won't match the S&P 500 on every past date. This dataset is for learning. Trading profits need their own entry and exit prices.


In [ ]:
%pip install -q "pandas>=2.1,<3" numpy requests yfinance lxml


## 1. Pick the dates

Keep these settings the same for everyone. We save the raw inputs and reuse them on later runs. Set `REFRESH_DATA = True` when the group wants to download a new batch.


In [ ]:
from pathlib import Path
from io import StringIO
import hashlib
import sys

import pandas as pd
import requests
import yfinance as yf

# Find the repo from its root or notebooks folder.
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "src/data/preparation.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Clone this repo first. The first cell shows how.")
sys.path.insert(0, str(ROOT))
from src.data.preparation import (
    FEATURES, prepare_price_panel, make_model_dataset, add_date_splits,
)

START = "2016-01-01"
END = "2026-01-01"  # excludes this date, so all of 2025 is included
REFRESH_DATA = False
TRAIN_END = "2022-12-31"
VALIDATION_END = "2023-12-31"
features = FEATURES.copy()

RAW_DIR = ROOT / "data/raw"
PROCESSED_DIR = ROOT / "data/processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CONSTITUENTS_FILE = RAW_DIR / f"sp500_{START}_{END}.csv"
PRICES_FILE = RAW_DIR / f"prices_{START}_{END}.csv"
USE_CACHE = (
    not REFRESH_DATA and CONSTITUENTS_FILE.exists() and PRICES_FILE.exists()
)


## 2. Get the stock list and prices

The first run downloads prices for 2016–2025 and needs internet. Later runs load the saved files. Stocks with failed downloads show up in the cleaning summary.


In [ ]:
if USE_CACHE:
    sp500 = pd.read_csv(CONSTITUENTS_FILE)
else:
    response = requests.get(
        "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
        headers={"User-Agent": "Mozilla/5.0"}, timeout=30,
    )
    response.raise_for_status()
    required = ["Symbol", "Security", "GICS Sector", "GICS Sub-Industry"]
    tables = pd.read_html(StringIO(response.text))
    sp500 = next((table for table in tables if set(required).issubset(table.columns)), None)
    if sp500 is None:
        raise ValueError("The S&P 500 table is missing columns we need.")
    sp500 = sp500[required].copy()
    sp500["Ticker"] = sp500["Symbol"].str.strip().str.replace(".", "-", regex=False)

metadata_columns = ["Ticker", "Security", "GICS Sector", "GICS Sub-Industry"]
sp500 = sp500.sort_values("Ticker").reset_index(drop=True)
tickers = sp500["Ticker"].tolist()
print("Stocks in this dataset:", len(tickers))
display(sp500.head())


In [ ]:
if USE_CACHE:
    raw_prices = pd.read_csv(
        PRICES_FILE, header=[0, 1], index_col=0, parse_dates=[0], float_precision="round_trip"
    )
    print("Loaded saved prices:", PRICES_FILE.name)
else:
    raw_prices = yf.download(
        tickers=tickers, start=START, end=END, interval="1d",
        auto_adjust=False, group_by="ticker", multi_level_index=True,
        keepna=True, ignore_tz=True, threads=True, progress=True,
    )
    if raw_prices is None or raw_prices.empty:
        raise ValueError("No prices came through. Check your connection and rerun this cell.")

if not isinstance(raw_prices.columns, pd.MultiIndex) or raw_prices.columns.nlevels != 2:
    raise ValueError("Yahoo Finance should give us ticker and price columns.")
if "Adj Close" not in raw_prices.columns.get_level_values(1):
    raise ValueError("Adj Close is missing. Keep auto_adjust=False.")

quote_columns = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
expected_columns = pd.MultiIndex.from_product(
    [tickers, quote_columns], names=["Ticker", "Field"]
)
raw_prices = raw_prices.reindex(columns=expected_columns).sort_index()
raw_prices.index = pd.to_datetime(raw_prices.index)
raw_prices.index.name = "Date"
if raw_prices.empty or raw_prices.index.has_duplicates:
    raise ValueError("Price dates are missing or repeated.")
if (raw_prices.index < pd.Timestamp(START)).any() or (raw_prices.index >= pd.Timestamp(END)).any():
    raise ValueError("Saved prices fall outside START and END.")

# Save both files after checking the download.
if not USE_CACHE:
    raw_prices.to_csv(PRICES_FILE)
    sp500.to_csv(CONSTITUENTS_FILE, index=False)

raw_panel = (
    raw_prices.stack(level="Ticker", future_stack=True)
    .rename_axis(["Date", "Ticker"])
    .reset_index()
)
raw_panel.columns.name = None
print("Price rows before cleaning:", len(raw_panel))


## 3. Clean the prices and build features

We check duplicates, missing or infinite values, positive prices and volume, and the daily High/Low range. Bad values become `NaN`, which means missing. We keep the dates so a next-day return still covers just one trading day.

Our nine features track returns, momentum, volatility, relative volume, daily price range, and intraday return. Rolling averages and volatility need a full window of data. The formulas and checks live in `src/data/preparation.py`.


In [ ]:
panel, cleaning_summary = prepare_price_panel(raw_panel, sp500[metadata_columns])
display(pd.Series(
    {key: value for key, value in cleaning_summary.items()
     if key not in {"invalid_reason_counts", "reason_counts_overlap"}},
    name="row count",
))
display(pd.Series(cleaning_summary["invalid_reason_counts"], name="flagged rows (one row can fail several checks)"))

valid_per_stock = panel.groupby("Ticker")["QuoteValid"].sum()
unavailable_tickers = valid_per_stock.index[valid_per_stock.eq(0)].tolist()
print("Stocks with no usable prices:", unavailable_tickers)


## 4. Build the model table

We keep stock-days with valid prices and all nine usable features. We decide this using only what was known by that day's close.

`target` is the adjusted close-to-close return for the next trading day. A return of `0.01` means 1%. `target_rank` compares returns on the same date: higher means better. Use it to check rankings, and keep it out of the model inputs.

Some rows won't have tomorrow's return yet. We keep them for predictions and drop their missing targets only when training or scoring.


In [ ]:
model_data = make_model_dataset(panel, labeled_only=False)
model_data = add_date_splits(model_data, TRAIN_END, VALIDATION_END)
columns = ["Date", "Ticker", "Split", "LabelEndDate", *features, "target", "target_rank"]
model_data = model_data[columns].copy()

if model_data.empty:
    raise ValueError("No rows have all nine features. Check the cleaning summary.")
if not {"train", "validation", "test"}.issubset(set(model_data["Split"])):
    raise ValueError("A train, validation, or test period is empty. Check the dates and price history.")
assert not model_data.duplicated(["Date", "Ticker"]).any()
assert model_data[features].notna().all().all()
display(model_data.groupby("Split").agg(rows=("Ticker", "size"), known_targets=("target", "count")))
display(model_data.head())


## 5. Split the data

- Train through 2022.
- Try model settings using 2023.
- Compare final results using 2024–2025.

All stocks on a date stay in the same split. A boundary day gets marked `purged` if its next-day return falls in the next period. We skip that day when training or scoring.

Use **only `features`** as model inputs. Fit scaling and other learned steps on training data only. Keep `target`, `target_rank`, `LabelEndDate`, and `Split` out of the inputs.


In [ ]:
train = model_data.loc[model_data["Split"].eq("train")].dropna(subset=["target"])
validation = model_data.loc[model_data["Split"].eq("validation")].dropna(subset=["target"])
test_signals = model_data.loc[model_data["Split"].eq("test")].copy()
test = test_signals.dropna(subset=["target"])

X_train, y_train = train[features], train["target"]
X_validation, y_validation = validation[features], validation["target"]
X_test, y_test = test[features], test["target"]
print("Rows with known returns: train / validation / test:", len(train), len(validation), len(test))
print("Test rows we can predict:", len(test_signals))


## 6. Save and share

The group leader runs this once and shares the CSV. The printed SHA256 is the file's fingerprint; matching fingerprints mean everyone has the same file. Generated data stays out of Git.

See `docs/backtesting.md` for our backtest plan. Keep the saved raw prices too: the backtester will need the next day's Open and Close.


In [ ]:
OUTPUT_FILE = PROCESSED_DIR / "model_data.csv"
model_data.to_csv(OUTPUT_FILE, index=False)
digest = hashlib.sha256()
with OUTPUT_FILE.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        digest.update(chunk)
checksum = digest.hexdigest()
print("Saved:", OUTPUT_FILE)
print("SHA256:", checksum)

# Everyone can load the shared CSV with this line:
# model_data = pd.read_csv(OUTPUT_FILE, parse_dates=["Date", "LabelEndDate"])
